In [1]:
import os
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

from tool import get_weather, calculate

load_dotenv()

True

## 🔧 Model and Tool Initialization  
This section sets up a minimal baseline agent with no guardrails or middleware applied.  
It serves as the “clean reference version” of the agent, allowing later guardrail-enhanced behaviors to be compared against the default behavior.

In [2]:
#model = init_chat_model("openai:gpt-4.1-mini")

model = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="azure_openai",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
    api_version="2025-01-01-preview",
)

In [3]:
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant. Use tools when needed",
)

In [4]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather of Delhi?"
            }
        ]
    }
)

print("\n ---- FINAL ANSWER ----")
print(result["messages"][-1].content)


 ---- FINAL ANSWER ----
The weather in Delhi is currently sunny.


## 🧩 Multi‑Tool Agent (No Memory, No Guardrails)  
This section demonstrates how the agent can invoke multiple tools (e.g., `get_weather`, `calculate`) within a single request.  
At this stage, the agent has no safety, memory, or reliability mechanisms enabled.

In [5]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="You are a helpful assistant. Use tools when needed",
)

In [6]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather in Delhi? and what's (3+5)*12?"
            }
        ]
    }
)

print("\n ---- FINAL ANSWER 1 ----")
print(result["messages"][-1].content)


 ---- FINAL ANSWER 1 ----
The weather in Delhi is described as always sunny. The result of the expression (3+5)*12 is 96.


In [7]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what city did I say I want to get the weather of ?"
            }
        ]
    }
)

print("\n ---- FINAL ANSWER 2 ----")
print(result["messages"][-1].content)


 ---- FINAL ANSWER 2 ----
You have not mentioned any city for which you want to get the weather. Please provide the name of the city.


## 🧠 Conversation Memory (Checkpointer)  
This section introduces `InMemorySaver`, enabling the agent to retain conversation state across turns.  
This is not a guardrail; it is a foundational capability for building stateful agents.

In [8]:
from langgraph.checkpoint.memory import InMemorySaver

In [9]:
checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_weather, calculate],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    checkpointer=checkpointer,
)

In [10]:
config = {"configurable": {"thread_id": "demo-thread-1"}}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is the weather in Delhi? and what's (3+5)*12?"
            }
        ]
    },
    config=config
)

print("\n ---- FINAL ANSWER 1 ----")
print(result["messages"][-1].content)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what city did I say I want to get the weather of ?"
            }
        ]
    },
    config=config
)

print("\n ---- FINAL ANSWER 2 ----")
print(result["messages"][-1].content)


 ---- FINAL ANSWER 1 ----
The weather in Delhi is described as always sunny. The calculation of (3+5)*12 is 96.

 ---- FINAL ANSWER 2 ----
You asked for the weather in the city of Delhi.


In [11]:
print("\n --- FULL MESSAGE TRACE ---")
for m in result["messages"]:
    print(type(m), getattr(m, "content", None))


 --- FULL MESSAGE TRACE ---
<class 'langchain_core.messages.human.HumanMessage'> what is the weather in Delhi? and what's (3+5)*12?
<class 'langchain_core.messages.ai.AIMessage'> 
<class 'langchain_core.messages.tool.ToolMessage'> It's always sunny in Delhi
<class 'langchain_core.messages.tool.ToolMessage'> 96
<class 'langchain_core.messages.ai.AIMessage'> The weather in Delhi is described as always sunny. The calculation of (3+5)*12 is 96.
<class 'langchain_core.messages.human.HumanMessage'> what city did I say I want to get the weather of ?
<class 'langchain_core.messages.ai.AIMessage'> You asked for the weather in the city of Delhi.


## 🧑‍💼 User Context (User‑Level Memory)  
This section demonstrates how user‑specific context (e.g., `user_id`) can be passed into the agent.  
This capability is essential for later guardrails such as preference memory.

In [12]:
from tool import get_weather, calculate, search_docs, Context, get_user_id

agent = create_agent(
        model=model,
        tools=[get_weather, calculate, search_docs, get_user_id],
        system_prompt="You are a helpful assistant. Use tools when needed",
        checkpointer=checkpointer,
    )

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "what is my user id?"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    config=config
)

## 🧱 Structured Output Guardrail (Output Safety)  
This section applies a Pydantic schema (`SupportActionPlan`) to enforce structured JSON output.  
This prevents format hallucinations, missing fields, and inconsistent responses.  
It is a form of **Output Safety Guardrail**.

In [13]:
from pydantic import BaseModel, Field

class SupportActionPlan(BaseModel):
    """A structured plan for resolving a support request."""

    summary: str = Field(description="1-2 sentence summary of the issue")
    steps: list[str] = Field(description="Concrete steps the user should take")
    needs_human: bool = Field(description="True if a human should review before action")

In [14]:
agent = create_agent(
        model=model,
        tools=[get_weather, calculate, search_docs, get_user_id],
        system_prompt="You are a helpful support assistant. Use tools when needed",
        checkpointer=checkpointer,
        response_format=SupportActionPlan
    )

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "I can't reset my password. What do I do ?"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    config=config
)

print("\n ---- FINAL ANSWER ----")
print(result["structured_response"])
print(result["messages"][-1].content)


 ---- FINAL ANSWER ----
summary='User is unable to reset their password and needs guidance.' steps=["Ensure you are using the correct 'Forgot Password' link on the login page.", 'Check your email inbox (and spam folder) for any password reset emails.', 'If you do not receive a reset email, try requesting the reset process again after some time.', 'Contact customer or technical support directly for further assistance if the problem persists.'] needs_human=True
{"summary":"User is unable to reset their password and needs guidance.","steps":["Ensure you are using the correct 'Forgot Password' link on the login page.","Check your email inbox (and spam folder) for any password reset emails.","If you do not receive a reset email, try requesting the reset process again after some time.","Contact customer or technical support directly for further assistance if the problem persists."],"needs_human":true}


## 📡 Streaming (Non‑Guardrail Feature)  
This section demonstrates the agent’s streaming capabilities.  
While not a guardrail, streaming is an important engineering feature for real‑time applications.

In [ ]:
config = {"configurable": {"thread_id": "demo-thread-1"}}

for mode, chunk in agent.stream(
    {"messages": [{"role": "user", "content": "Search docs for API limits and summarize."}]},
    stream_mode=["updates", "messages"],
    config=config
):
    if mode == "messages":
        token, metadata = chunk
        if token.content:  
            print(token.content, end="", flush=True)
    elif mode == "updates":
        # You can inspect node-level updates here
        pass

No relevant KB entry found.{"summary":"No documentation found regarding API limits.","steps":["Check if there is any updated or alternate documentation source on API limits.","Contact support or the API provider to request information about API limits.","Review your API usage dashboard, if available, for any limit notifications or guidelines."],"needs_human":true}

## 🎛️ Preference Memory Guardrail (Behavior Consistency)  
This section introduces a memory store that allows the agent to save and recall user preferences  
(e.g., preferred response style).  
This ensures consistent behavior across turns and is considered a **Behavior Consistency Guardrail**.

In [16]:
from langgraph.store.memory import InMemoryStore
from langchain.tools import tool, ToolRuntime

store = InMemoryStore()

@tool
def save_preference(style: str, runtime: ToolRuntime[Context]) -> str:
    """Save the user's preference response style."""
    store = runtime.store
    store.put(("preferences",), runtime.context.user_id, {"style": style})
    return "Saved."

@tool
def read_preference(runtime: ToolRuntime[Context]) -> str:
    """Read the user's preference style."""
    pref = runtime.store.get(("preferences",), runtime.context.user_id)
    return pref.value.get("style", "balanced") if pref else "balanced"

In [17]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculate, search_docs, get_user_id, save_preference, read_preference],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    # checkpointer=checkpointer,
    # response_format=SupportActionPlan,
    store=store,
    # middleware=[
    #     SummarizationMiddleware(
    #         model="openai:gpt-4.1-mini",
    #         trigger=("tokens", 4000),
    #         keep=("messages", 20)
    #     )
    # ]
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My Style is: super concise."
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What style do i prefer?"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

print("\n ---- FINAL ANSWER ----")
# print(result["structured_response"])
print(result["messages"][-1].content)


 ---- FINAL ANSWER ----
Your preferred response style is super concise.


## 🔍 Logging Middleware (Observability)  
This section adds `before_model` and `after_model` middleware hooks to log model inputs and outputs.  
These are not guardrails; they are **observability tools** that improve debuggability and traceability.

In [18]:
from langchain.agents.middleware import AgentState, before_model, after_model

from langgraph.runtime import Runtime

@before_model
def log_before_model(state: AgentState, runtime: Runtime):
    last = state["messages"][-1]
    print(f"[before_model] last message: {getattr(last, 'content', last)}")
    return None


@after_model
def log_after_model(state: AgentState, runtime: Runtime):
    last = state["messages"][-1]
    print(f"[after_model] model output: {getattr(last, 'content', last)}")
    return None

In [19]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculate, search_docs, get_user_id, save_preference, read_preference],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    # checkpointer=checkpointer,
    # response_format=SupportActionPlan,
    store=store,
    # middleware=[
    #     SummarizationMiddleware(
    #         model="openai:gpt-4.1-mini",
    #         trigger=("tokens", 4000),
    #         keep=("messages", 20)
    #     )
    # ]
    middleware=[
        log_after_model,
        log_before_model,
    ]
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My Style is: super concise."
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What style do i prefer?"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

print("\n ---- FINAL ANSWER ----")
# print(result["structured_response"])
print(result["messages"][-1].content)

[before_model] last message: My Style is: super concise.
[after_model] model output: 
[before_model] last message: Saved.
[after_model] model output: Got it. Style set to super concise.
[before_model] last message: What style do i prefer?
[after_model] model output: 
[before_model] last message: super concise
[after_model] model output: You prefer a super concise response style. How can I assist you further?

 ---- FINAL ANSWER ----
You prefer a super concise response style. How can I assist you further?


## 🔁 Retry & Dynamic Prompt (Reliability Middleware)  
This section introduces two reliability mechanisms:  
- **Retry Middleware**: Automatically retries model calls on transient failures.  
- **Dynamic Prompt Middleware**: Adjusts the system prompt based on conversation length.  
These are engineering reliability features, not guardrails.

In [21]:
from typing import Callable

from langchain.agents.middleware import AgentState, before_model, after_model, wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt

@wrap_model_call
def retry_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as ex:
            if attempt == 2:
                print(f"[wrap_model_call] retry {attempt + 1}/3 after error: {ex}")
                raise
            print(f"[wrap_model_call] retry {attempt + 1}/3 after error: {ex}")


@dynamic_prompt
def system_prompt_from_context(request: ModelRequest) -> str:
    # Example: be extra concise for long conversation
    if len(request.messages) > 10:
        return "You are a helpful assistant. Be extremely concise."
    return "You are a helpful assistant."

In [25]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculate, search_docs, get_user_id, save_preference, read_preference],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    # checkpointer=checkpointer,
    # response_format=SupportActionPlan,
    store=store,
    # middleware=[
    #     SummarizationMiddleware(
    #         model="openai:gpt-4.1-mini",
    #         trigger=("tokens", 4000),
    #         keep=("messages", 20)
    #     )
    # ]
    middleware=[
        log_after_model,
        log_before_model,
        system_prompt_from_context,
        retry_model
    ]
)

agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "My Style is: super concise."
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "can you calculate 7*5*5, can you find me the best hotel in my area with 500 budget"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

print("\n ---- FINAL ANSWER ----")
# print(result["structured_response"])
print(result["messages"][-1].content)

[before_model] last message: My Style is: super concise.
[after_model] model output: 
[before_model] last message: Saved.
[after_model] model output: Got it. Your style is set to super concise. How can I assist you?
[before_model] last message: can you calculate 7*5*5, can you find me the best hotel in my area with 500 budget
[after_model] model output: 
[before_model] last message: 175
[after_model] model output: The result of 7 * 5 * 5 is 175. 

Regarding finding the best hotel in your area with a budget of 500, could you please specify your location or area? This will help me search for hotels accordingly.

 ---- FINAL ANSWER ----
The result of 7 * 5 * 5 is 175. 

Regarding finding the best hotel in your area with a budget of 500, could you please specify your location or area? This will help me search for hotels accordingly.


## 🚫 Content Filter Guardrail (Input Safety)  
This section demonstrates a custom rule‑based content filter that blocks requests containing banned keywords.  
When triggered, the agent halts execution and returns a safe fallback message.  
This is an **Input Safety Guardrail**.

In [ ]:
from typing import Callable, Any

from langchain.agents.middleware import AgentState, before_model, after_model, \
    wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt, PIIMiddleware, \
    AgentMiddleware, hook_config

class ContentFilterMiddleware(AgentMiddleware):
    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned = [b.lower() for b in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None
        content = getattr(state["messages"][0], "content", "").lower()
        if any(b in content for b in self.banned):
            return {
                "messages": [{"role": "assistant", "content": "I can't help you with that request. Please rephrase."}],
                "jump_to": "end"
            }

        return None

## 🔐 PII Guardrail (Input Sanitization)  
This section applies PII protection middleware that automatically redacts or masks sensitive information  
(e.g., email addresses, credit card numbers) before the model processes the input.  
This is an **Input Sanitization Guardrail**.

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_weather, calculate, search_docs, get_user_id, save_preference, read_preference],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    # checkpointer=checkpointer,
    # response_format=SupportActionPlan,
    store=store,
    # middleware=[
    #     SummarizationMiddleware(
    #         model="openai:gpt-4.1-mini",
    #         trigger=("tokens", 4000),
    #         keep=("messages", 20)
    #     )
    # ]
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        ContentFilterMiddleware(["hotel"]),
    ]
)

agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": "My Style is: super concise."
                }
            ]
        },
        context=Context(user_id="raj713335"),
        # config=config
    )

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "can you calculate 7*5*5, can you find me the best hotel in my area with 500 budget"
            }
        ]
    },
    context=Context(user_id="raj713335"),
    # config=config
)

print("\n ---- FINAL ANSWER ----")
# print(result["structured_response"])
print(result["messages"][-1].content)


 ---- FINAL ANSWER ----
I can't help you with that request. Please rephrase.


## 🛑 Human‑in‑the‑Loop (HITL) Guardrail (Action Safety)  
This section introduces HITL middleware that intercepts sensitive tool calls  
(e.g., `send_email`) and pauses execution until a human reviewer approves, edits, or rejects the action.  
This is an **Action Safety Guardrail**, ensuring the agent cannot autonomously perform high‑risk operations.

In [ ]:
from langchain.agents.middleware import AgentState, before_model, after_model, \
    wrap_model_call, ModelRequest, ModelResponse, dynamic_prompt, PIIMiddleware, \
    AgentMiddleware, hook_config, HumanInTheLoopMiddleware

from langgraph.types import Command

from tool import get_weather, calculate, search_docs, Context, get_user_id, send_email

checkpointer = InMemorySaver()

agent = create_agent(
    model=model,
    tools=[get_weather, calculate, search_docs, get_user_id, save_preference, read_preference, send_email],
    system_prompt="You are a helpful support assistant. Use tools when needed",
    checkpointer=checkpointer,
    # response_format=SupportActionPlan,
    store=store,
    # middleware=[
    #     SummarizationMiddleware(
    #         model="openai:gpt-4.1-mini",
    #         trigger=("tokens", 4000),
    #         keep=("messages", 20)
    #     )
    # ]
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        ContentFilterMiddleware(["hotel"]),
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "search_docs": False
            }
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "hitl-accept-demo"}}

result = agent.invoke({"messages": [
    {"role": "user", "content": "Email the team: subject 'Update, body 'All Good Folks!'"}
]},
    context=Context(user_id="raj713335"),
    config=config
)

print("INTERRUPT:", result.get("__interrupt__"))

result = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}),
    config=config
)

print("\n ---- FINAL ANSWER ----")
print(result["messages"][-1].content)

INTERRUPT: [Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'team', 'subject': 'Update', 'body': 'All Good Folks!'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'team', 'subject': 'Update', 'body': 'All Good Folks!'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='09405a549f5ae37c84b50c18e8038653')]

 ---- FINAL ANSWER ----
The email with the subject "Update" and the body "All Good Folks!" has been sent to the team. Is there anything else you would like to do?


In [ ]:
config = {"configurable": {"thread_id": "hitl-reject-demo"}}

result = agent.invoke({"messages": [
    {"role": "user", "content": "Email the team: subject 'Update, body 'All Good Folks!'"}
]},
    context=Context(user_id="raj713335"),
    config=config
)

print("INTERRUPT:", result.get("__interrupt__"))

result = agent.invoke(Command(resume={"decisions": [{"type": "reject"}]}),
    config=config
)

print("\n ---- FINAL ANSWER ----")
print(result["messages"][-1].content)

INTERRUPT: [Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'to': 'team', 'subject': 'Update', 'body': 'All Good Folks!'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'to': 'team', 'subject': 'Update', 'body': 'All Good Folks!'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='9cd2af910680b4d83a02148a151796ed')]

 ---- FINAL ANSWER ----
I noticed that you rejected the email sending action. Would you like me to send the email to the team now?
